In [184]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [115]:
data_rav = pd.read_csv('C:\\Users\\sashi\\OneDrive\\Desktop\\SpeechEmotionRecognition\\ravdess_features.csv')

data_rav = data_rav.dropna()

data_crem = pd.read_csv("C:/Users/sashi/Downloads/crema_d_features_cleaned.csv")

data_crem = data_crem.dropna()

data_combined = pd.concat([data_rav, data_crem], join = 'inner',ignore_index=True)

data_combined['emotion'] = data_combined['emotion'].replace({
    1: 3,
    2: 6,
    3: 1,
    4: 2,
    5: 0,
    6: 5,
    7: 4,
    8: 7,
})

print(data_combined.columns)
print(data_combined.head())








Index(['filename', 'emotion', 'intensity', 'duration', 'sample_rate',
       'zcr_mean', 'zcr_std', 'spectral_centroid_mean',
       'spectral_centroid_std', 'spectral_rolloff_mean',
       'spectral_rolloff_std', 'spectral_bandwidth_mean',
       'spectral_bandwidth_std', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean',
       'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std',
       'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean',
       'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std',
       'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std',
       'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std',
       'chroma_1_mean', 'chroma_1_std', 'chroma_2_mean', 'chroma_2_std',
       'chroma_3_mean', 'chroma_3_std', 'chroma_4_mean', 'chroma_4_std',
       'chroma_5_mean', 'chroma_5_std', 'chroma_6_mean', 'chroma_6_std',
       'chroma_7_mean', 'chroma_7_std', 'chroma_8_mean', 'chroma_8_std',
       'chroma

In [116]:
emotion_map = {
    'Anger': 0,
    'Happy': 1,
    'Sad': 2,
    'Neutral': 3,
    'Disgust': 4,
    'Fear': 5
    
}

data_combined['Emotion_num'] = data_combined['emotion'].map(emotion_map)

data_combined.loc[0:5759, 'Emotion_num'] = data_combined.loc[0:5759, 'emotion']


print(data_combined.loc[5759:6000, ['emotion', 'Emotion_num']])

      emotion Emotion_num
5759        7           7
5760    Anger         0.0
5761    Anger         0.0
5762  Neutral         3.0
5763  Neutral         3.0
...       ...         ...
5996  Disgust         4.0
5997  Disgust         4.0
5998      Sad         2.0
5999    Happy         1.0
6000    Happy         1.0

[242 rows x 2 columns]


C:\Users\sashi\AppData\Local\Temp\ipykernel_52216\2522198600.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3 3 3 ... 7 7 7]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_combined.loc[0:5759, 'Emotion_num'] = data_combined.loc[0:5759, 'emotion']


In [120]:
data_combined['tempo'] = (
    data_combined['tempo']
    .astype(str)
    .str.replace('[\[\]]', '', regex=True)   # remove [ and ]
    .astype(float)
)
data_combined.select_dtypes(exclude=['number']).columns


<>:4: SyntaxWarning: invalid escape sequence '\['
<>:4: SyntaxWarning: invalid escape sequence '\['
C:\Users\sashi\AppData\Local\Temp\ipykernel_52216\1562352221.py:4: SyntaxWarning: invalid escape sequence '\['
  .str.replace('[\[\]]', '', regex=True)   # remove [ and ]


Index(['filename', 'emotion', 'intensity', 'Emotion_num'], dtype='object')

In [125]:
X = data_combined[['zcr_mean', 'zcr_std', 'spectral_centroid_mean',
       'spectral_centroid_std', 'spectral_rolloff_mean',
       'spectral_rolloff_std', 'spectral_bandwidth_mean',
       'spectral_bandwidth_std', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean',
       'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std',
       'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean',
       'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std',
       'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std',
       'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std',
       'chroma_1_mean', 'chroma_1_std', 'chroma_2_mean', 'chroma_2_std',
       'chroma_3_mean', 'chroma_3_std', 'chroma_4_mean', 'chroma_4_std',
       'chroma_5_mean', 'chroma_5_std', 'chroma_6_mean', 'chroma_6_std',
       'chroma_7_mean', 'chroma_7_std', 'chroma_8_mean', 'chroma_8_std',
       'chroma_9_mean', 'chroma_9_std', 'chroma_10_mean', 'chroma_10_std',
       'chroma_11_mean', 'chroma_11_std', 'chroma_12_mean', 'chroma_12_std',
       'tempo', 'rms_mean', 'rms_std']
].to_numpy()


print("X Shape and head: ", X.shape, X[:5,:])

Y = data_combined[['Emotion_num']]

Y = Y.to_numpy().reshape(-1,1)

print("Y Shape and head: ", Y.shape, Y[:5,:])


X Shape and head:  (13199, 61) [[ 3.72712249e-01  2.67765638e-01  3.47093769e+03  1.73231585e+03
   6.33038526e+03  3.00290746e+03  2.62824358e+03  7.46399668e+02
  -6.97792600e+02  1.83030440e+02  5.48900400e+01  7.21684800e+01
   6.63465500e-01  1.91957990e+01  1.24357860e+01  2.09307540e+01
   7.73395060e+00  1.73393100e+01  5.30750300e-01  1.37613380e+01
  -3.21663120e+00  1.08533890e+01 -3.15939430e+00  1.15054540e+01
  -1.09775510e+01  1.70198140e+01 -2.84871100e+00  8.86510000e+00
   8.15297400e-01  8.66975100e+00 -3.03706670e+00  9.67813600e+00
   1.95544650e+00  9.78021300e+00  6.55044700e-01  2.96440240e-01
   6.10740900e-01  2.91287870e-01  5.68578660e-01  3.28646030e-01
   5.72311640e-01  3.00669670e-01  5.55372360e-01  2.89184600e-01
   5.29292800e-01  2.85663520e-01  5.87064500e-01  3.02491200e-01
   6.38354960e-01  2.90045920e-01  6.42917900e-01  2.75507200e-01
   6.06422250e-01  2.62522400e-01  6.10109200e-01  2.96289950e-01
   6.04038360e-01  2.94712900e-01  8.07495117

In [126]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("X scaled Shape and head: ", X.shape, X[:5,:])


X scaled Shape and head:  (13199, 61) [[ 2.12967169  1.61204136  1.73105294  1.44893222  1.55143953  1.40901163
   1.29582933  0.97465994 -1.74958242  1.14433463 -0.9138394   1.98293768
  -0.08577294 -1.07878535 -0.82462129  0.3248046   1.42264356 -0.35639302
  -0.5798879  -0.54957269  1.02978721 -1.01174573 -0.38631111 -0.16570899
  -0.26243853  1.51544097 -0.79030182 -0.10043065  0.82104185 -0.029025
   0.31467267  0.54367182  0.16776631  0.75313534  2.66818891  0.16675889
   2.11601624 -0.05678454  1.78744048  0.90330914  1.65313473  0.33587121
   1.39109226  0.00498499  1.11318506 -0.1029492   1.59034464  0.14481122
   2.0135319  -0.22933249  1.98203374 -0.84648689  0.94528386 -1.76131153
   0.96012532 -0.78069718  1.83328214  0.05513416 -1.1970802  -0.73194216
  -0.67178503]
 [ 2.20327068  1.64816711  1.4785981   1.0472863   1.48584727  1.21086347
   1.40947893  0.8630421  -1.71077629  1.18236433 -0.90306862  1.63096244
  -0.2497519  -1.06931243 -0.66978615 -0.04693174  1.53262809

In [129]:
encoder = OneHotEncoder(sparse_output=False)  # gives a dense array
Y_onehot = encoder.fit_transform(Y)


In [130]:
x_train, x_test, y_train, y_test = train_test_split(X, Y_onehot, test_size = 0.25, random_state = 0)


In [132]:
x_train, x_test, y_train, y_test = train_test_split(X, Y_onehot, test_size=0.25, random_state=42)
print ("xtrain.shape is ",x_train.shape)
print ("xtest.shape is ",x_test.shape)

print ("ytrain.shape is ", y_train.shape)
print ("ytest.shape is ", y_test.shape)

xtrain.shape is  (9899, 61)
xtest.shape is  (3300, 61)
ytrain.shape is  (9899, 8)
ytest.shape is  (3300, 8)


In [187]:
X_train, X_val, Y_train, Y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

In [193]:
model = models.Sequential()
model.add(layers.Conv1D(128, 1, input_shape=(X.shape[1], 1)))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Conv1D(128, 1))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.MaxPooling1D((2)))

model.add(layers.Conv1D(128, 1))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.MaxPooling1D((2)))

model.add(layers.Conv1D(64, 1))
model.add(layers.BatchNormalization())
model.add(layers.Activation('relu'))

model.add(layers.Flatten())
model.add(layers.Dense(500, activation='relu'))
model.add(layers.Dense(500, activation='relu'))
model.add(layers.Dropout(0.25))

model.add(layers.Dense(Y_onehot.shape[1], activation='softmax'))

model.summary()

Model: "sequential_46"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_92 (Conv1D)              │ (None, 61, 128)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 61, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 61, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_93 (Conv1D)              │ (None, 61, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 61, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 61, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_53 (MaxPooling1D) │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_94 (Conv1D)              │ (None, 30, 128)        │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_54 (MaxPooling1D) │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_95 (Conv1D)              │ (None, 15, 64)         │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 15, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_32 (Flatten)            │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_69 (Dense)                │ (None, 500)            │       480,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_70 (Dense)                │ (None, 500)            │       250,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_71 (Dense)                │ (None, 8)              │         4,008 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 778,336 (2.97 MB)

 Trainable params: 777,440 (2.97 MB)

 Non-trainable params: 896 (3.50 KB)

In [198]:
adam = optimizers.Adam(learning_rate=0.0005)  # default is 0.001
model.compile(optimizer=adam,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [199]:
early_stop = EarlyStopping(
    monitor='val_loss',      
    patience=10,             
    restore_best_weights=True)

history = model.fit(X_train, Y_train, epochs=20, 
                    validation_data=(X_val, Y_val), callbacks=[early_stop])

Epoch 1/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.3704 - loss: 1.5977 - val_accuracy: 0.2116 - val_loss: 1.9718
Epoch 2/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.5065 - loss: 1.2719 - val_accuracy: 0.4586 - val_loss: 1.3691
Epoch 3/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.6032 - loss: 1.0394 - val_accuracy: 0.5843 - val_loss: 1.1160
Epoch 4/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.6699 - loss: 0.8666 - val_accuracy: 0.6207 - val_loss: 0.9739
Epoch 5/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.7232 - loss: 0.7396 - val_accuracy: 0.6348 - val_loss: 0.9771
Epoch 6/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.7586 - loss: 0.6318 - val_accuracy: 0.6490 - val_loss: 0.9903
Epoch 7/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.8001 - loss: 0.5402 - val_accuracy: 0.6485 - val_loss: 1.0187
Epoch 8/20
248/248 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.8386 - loss: 0.4378 - val_acc

In [200]:
print((history.history['val_accuracy']))

[0.2116161584854126, 0.45858585834503174, 0.584343433380127, 0.620707094669342, 0.6348484754562378, 0.6489899158477783, 0.6484848260879517, 0.6434343457221985, 0.6196969747543335, 0.6348484754562378, 0.6409090757369995, 0.65505051612854, 0.653030276298523, 0.6489899158477783]
